# Harmonized 72-Class Leaf Disease Model Comparison

This notebook rebuilds the canonical 70/15/15 benchmark from the attached Kaggle dataset, then trains the supported mobile classifiers and compares full INT8 TFLite quality, model size, and single-thread CPU latency. CPU latency is a proxy; validate finalists on the target Android device.

Attach the multi-crop leaf disease dataset in Kaggle Notebook settings and enable a GPU accelerator.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
BRANCH = 'ut'
REPO_DIR = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
elif not os.path.isfile(os.path.join(REPO_DIR, 'training', 'compare_models.py')):
    raise RuntimeError('Existing repository checkout is not the updated ut pipeline; restart the Kaggle session and rerun.')
print('Repository:', REPO_DIR, 'branch:', BRANCH)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(REPO_DIR, 'requirements.txt')], check=True)
print('Dependencies installed')

In [ ]:
from pathlib import Path
sys.path.insert(0, REPO_DIR)
from dataset.taxonomy import CANONICAL_CLASSES, canonicalize_label

def has_expected_labels(folder):
    if not folder.is_dir():
        return False
    labels = [p.name for p in folder.iterdir() if p.is_dir()]
    mapped = set()
    try:
        mapped = {canonicalize_label(label) for label in labels}
    except ValueError:
        return False
    return len(mapped) == 72

def find_dataset_source():
    candidates = []
    for mount in sorted(Path('/kaggle/input').iterdir()):
        if not mount.is_dir():
            continue
        candidates.extend([mount / 'dataset' / 'raw', mount / 'raw', mount / 'dataset' / 'processed', mount / 'processed', mount / 'dataset', mount])
    for candidate in candidates:
        split_dirs = [candidate / name for name in ('train', 'val', 'test')]
        if all(path.is_dir() for path in split_dirs) and all(has_expected_labels(path) for path in split_dirs):
            return candidate
        if has_expected_labels(candidate):
            return candidate
    raise FileNotFoundError('Could not find a source root containing the canonical 72 classes under /kaggle/input.')

SOURCE_DIR = find_dataset_source()
BENCHMARK_DIR = Path('/kaggle/working/harmonized_benchmark')
print('Source:', SOURCE_DIR)
print('Canonical classes:', len(CANONICAL_CLASSES))

In [ ]:
if BENCHMARK_DIR.exists():
    raise FileExistsError(f'Benchmark output already exists: {BENCHMARK_DIR}. Restart the session or change BENCHMARK_DIR.')
subprocess.run([sys.executable, os.path.join(REPO_DIR, 'dataset', 'prepare_benchmark.py'), '--data-dir', str(SOURCE_DIR), '--output-dir', str(BENCHMARK_DIR), '--seed', '42'], check=True)
print('Prepared benchmark at', BENCHMARK_DIR)

In [ ]:
COMPARISON_DIR = Path('/kaggle/working/leaf_model_comparison')
if COMPARISON_DIR.exists():
    raise FileExistsError(f'Comparison output already exists: {COMPARISON_DIR}. Restart the session or set a new output path.')
subprocess.run([sys.executable, os.path.join(REPO_DIR, 'training', 'compare_models.py'), '--data-dir', str(BENCHMARK_DIR), '--output-dir', str(COMPARISON_DIR)], check=True)
print('Comparison report:', COMPARISON_DIR / 'model_comparison.json')

In [ ]:
import json
import pandas as pd

report = json.loads((COMPARISON_DIR / 'model_comparison.json').read_text())
display(pd.read_csv(COMPARISON_DIR / 'model_comparison.csv').sort_values(['int8_macro_f1', 'int8_mean_inference_ms'], ascending=[False, True]))
print('Pareto frontier:', report['pareto_frontier'])
print(report['latency_note'])